# PCT Training - Occlusion

Trains Menghao PCT model from the Point-Transformers implementation: https://github.com/qq456cvb/Point-Transformers
Changes: Added an occlusion functions that randomly drops between 0.1 to 0.9 points from the cloud in a plane-based manner

Data: fullmodelnet40 version

## Env prep

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# Paths, folders
import os

REPO_PATH = '/content/pointcloud-bench'
DRIVE_PATH = '/content/drive/MyDrive/pointcloud-bench'
results_dir = os.path.join(DRIVE_PATH, 'results')
os.makedirs(results_dir, exist_ok=True)

In [ ]:
# Get repo
!git clone --recurse-submodules --branch pct-plots https://github.com/DavidClaszen/pointcloud-bench {REPO_PATH}

# Submodule handling
%cd {REPO_PATH}
!git submodule update --init --recursive
%cd repos/Point-Transformers
!git fetch origin pct-plots
!git checkout pct-plots
!git pull origin pct-plots
%cd {REPO_PATH}

%pip install -r envs/pct/requirements.txt

Cloning into '/content/pointcloud-bench'...
remote: Enumerating objects: 252, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 252 (delta 43), reused 71 (delta 33), pack-reused 157 (from 1)
Receiving objects: 100% (252/252), 8.87 MiB | 30.49 MiB/s, done.
Resolving deltas: 100% (95/95), done.
Submodule 'repos/PAPNet' (https://github.com/DavidClaszen/PAPNet.git) registered for path 'repos/PAPNet'
Submodule 'repos/Point-Transformers' (https://github.com/DavidClaszen/Point-Transformers.git) registered for path 'repos/Point-Transformers'
Cloning into '/content/pointcloud-bench/repos/PAPNet'...
remote: Enumerating objects: 133, done.        
remote: Counting objects: 100% (133/133), done.        
remote: Compressing objects: 100% (105/105), done.        
remote: Total 133 (delta 55), reused 85 (delta 27), pack-reused 0 (from 0)        
Receiving objects: 100% (133/133), 9.66 MiB | 9.79 MiB/s, done.
Resolving deltas: 100% (55/5

In [5]:
# Check for CUDA/GPU
import torch, sys
print(sys.version)
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.is_available())

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.3.1+cu121 CUDA: 12.1 GPU: True


In [6]:
# Copy and unzip only the fullmodelnet40 set
# Evaluation will be done in other notebook
!rsync -avP {DRIVE_PATH}/datasets/fullmodelnet40.tar.gz {REPO_PATH}/datasets
%cd {REPO_PATH}
!tar -xvzf datasets/fullmodelnet40.tar.gz -C datasets

sending incremental file list
fullmodelnet40.tar.gz
  1,833,210,387 100%   71.67MB/s    0:00:24 (xfr#1, to-chk=0/1)

sent 1,833,658,058 bytes  received 35 bytes  71,908,160.51 bytes/sec
total size is 1,833,210,387  speedup is 1.00
/content/pointcloud-bench
fullmodelnet40/
fullmodelnet40/test_filenames.txt
fullmodelnet40/test_gt_rot.npy
fullmodelnet40/test_gt_tra.npy
fullmodelnet40/test_labels.npy
fullmodelnet40/test_points.npy
fullmodelnet40/train_filenames.txt
fullmodelnet40/train_gt_rot.npy
fullmodelnet40/train_gt_tra.npy
fullmodelnet40/train_labels.npy
fullmodelnet40/train_points.npy


# Model Training

Since we're only using PAPNet style data here, always set `use_papnet_loader` to `True`.
Model trained on partial50 is still valid, doesn't have to be trained again, but we do need one trained on the new full PAPNet.


In [7]:
%cd /content/pointcloud-bench/repos/Point-Transformers
!python train_cls.py --help

/content/pointcloud-bench/repos/Point-Transformers
train_cls is powered by Hydra.

== Configuration groups ==
Compose your configuration from those groups (group=option)

model: Hengshuang, Menghao, Nico


== Config ==
Override anything in the config (foo.bar=value)

model:
  name: Menghao
batch_size: 16
epoch: 200
learning_rate: 0.001
gpu: 0
num_point: 1024
optimizer: Adam
weight_decay: 0.0001
normal: true
use_papnet_loader: false
workers: 2
step_size: 50
data_path: ../../datasets/modelnet40_normal_resampled/
checkpoint_path: best_model.pth
partiality: ''


Powered by Hydra (https://hydra.cc)
Use --hydra-help to view Hydra specific help




In [ ]:
# Train Menghao
!python train_cls.py model=Menghao use_papnet_loader=True batch_size=256 learning_rate=0.0005 epoch=50 workers=4 step_size=5 data_path=../../datasets/fullmodelnet40/ occlusion=True occlusion_min=0.1 occlusion_max=0.9

model:
  name: Menghao
batch_size: 256
epoch: 50
learning_rate: 0.0005
gpu: 0
num_point: 1024
optimizer: Adam
weight_decay: 0.0001
normal: true
use_papnet_loader: true
workers: 4
step_size: 5
data_path: ../../datasets/fullmodelnet40/
checkpoint_path: best_model.pth
partiality: ''

[2025-11-30 05:18:57,918][__main__][INFO] - Load dataset ...
The size of train data is 98430
The size of test data is 2468
[2025-11-30 05:18:58,861][__main__][INFO] - No existing model, starting training from scratch...
[2025-11-30 05:19:02,185][__main__][INFO] - Start training...
[2025-11-30 05:19:02,185][__main__][INFO] - Epoch 1 (1/50):
100% 385/385 [02:38<00:00,  2.43it/s]
[2025-11-30 05:21:40,476][__main__][INFO] - Train Instance Accuracy: 0.337694
100% 10/10 [00:03<00:00,  2.55it/s]
[2025-11-30 05:21:44,455][__main__][INFO] - Test Instance Accuracy: 0.425934, Class Accuracy: 0.329506
[2025-11-30 05:21:44,455][__main__][INFO] - Best Instance Accuracy: 0.425934, Class Accuracy: 0.329506
[2025-11-30 05:21:

In [11]:
# Zip Point-Transformers logs, included last best model
!zip -r results.zip ./log/cls/Menghao/

  adding: log/cls/Menghao/ (stored 0%)
  adding: log/cls/Menghao/model.py (deflated 76%)
  adding: log/cls/Menghao/train_cls.log (deflated 87%)
  adding: log/cls/Menghao/best_model.pth (deflated 9%)
  adding: log/cls/Menghao/.hydra/ (stored 0%)
  adding: log/cls/Menghao/.hydra/config.yaml (deflated 29%)
  adding: log/cls/Menghao/.hydra/overrides.yaml (deflated 19%)
  adding: log/cls/Menghao/.hydra/hydra.yaml (deflated 66%)
